#### https://github.com/HVL-ML/DAT255/blob/main/notebooks/15_gradio_and_streamlit.ipynb er brukt som utgangspunkt i denne notebooken.

# Gradio (and Streamlit) deployment

For deploying an ML-based app there are various approaches, but [Gradio](gradio.app) and [Streamlit](https://streamlit.io/) are both quick and convenient ways to do so. Typically we would implement this in a plain python (`.py`) file rather than a `.ipynb` notebook, but Gradio works here too, so let's try that first. Streamlit needs to be run directly in python, but the code is given below, so you can try out that too.

In this example we set up an image classifier, where the user can upload an image and get the top 5 class predictions in return.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.applications.mobilenet_v2 import (
    MobileNetV2,
    preprocess_input,
    decode_predictions,
)
from PIL import Image

2026-03-27 16:13:46.752123: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-27 16:13:46.760538: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774624426.770607    9267 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774624426.773892    9267 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-27 16:13:46.784637: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

## Set up a pretrained model

Let's download and set up a `MobileNetV2` model, trained on the 1000 classes in the ImageNet dataset. You can change this to anything you like. Remeber, however, to also use the appropriate preprocessing function.

In [2]:
# model = MobileNetV2(weights="imagenet")

# The model loading below is from Deep Learning with Python, Third edition. Chapter: 8, Fitting the model
model1 = keras.models.load_model(
    "../models/augment.keras"
)
# The model loading below is from Deep Learning with Python, Third edition. Chapter: 8, Fitting the model
model2 = keras.models.load_model(
    "../models/train_hierarchical_cnn_global_aug_v2.keras"
)
# The model loading below is from Deep Learning with Python, Third edition. Chapter: 8, Fitting the model
model3 = keras.models.load_model(
    "../models/hierarchical_cnn_baseline_hybrid_crop.keras"
)

IMG_RES = (300, 300)

I0000 00:00:1774624429.173413    9267 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9022 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080, pci bus id: 0000:01:00.0, compute capability: 8.6


# Cellene under kommer tilnærmet direkte fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]

#### The two cells below are based on Deep Inside Convolutional Networks: Visualising Image Classification Models and Saliency Maps, Image-Specific Class Saliency Visualisation (https://arxiv.org/abs/1312.6034)

In [3]:
# This method is based on Deep Learning with Python, Third edition. Chapter: 2, Gradient computation,
# and get_gradients from https://keras.io/examples/vision/integrated_gradients/
def get_gradients(img_array, lvl, model, lvl2_pred_index):
    # img_array /= 255.0
    with tf.GradientTape() as tape:
        tape.watch(img_array)
        # Denne veiledningen på reduce_max ble brukt: https://www.tensorflow.org/api_docs/python/tf/math/reduce_max
        # print(img_array)
        if lvl2_pred_index != None:
            score = model(img_array)[lvl][0][lvl2_pred_index]
        else:
            score = tf.math.reduce_max(model(img_array)[lvl])
    gradients = tape.gradient(score, img_array)
    return gradients

In [4]:
def get_heatmap(img_array, lvl, model, lvl2_pred_index=None):
    gradients = get_gradients(img_array, lvl, model, lvl2_pred_index)
    abs_gradients = np.abs(gradients[0])
    # np.max is from: Deep Learning with Python, Third edition. Chapter: 10, Displaying the class activation heatmap
    max_gradient = np.max(abs_gradients)
    pixel_gradients = ((abs_gradients / max_gradient) * 255.0)

    # Denne dokumentasjonen er brukt for å lage linjen under: https://numpy.org/doc/2.2/reference/generated/numpy.max.html
    return np.max(pixel_gradients, 2)

#### The idea of the method below is from: SmoothGrad: removing noise by adding noise, Capping outlying values (https://arxiv.org/pdf/1706.03825)

In [5]:
import math

def cap_heatmap(heatmap):
    values = []
    for i in range(len(heatmap)):
        for j in range(len(heatmap[i])):
            values.append(heatmap[i][j])
    values.sort()

    ninty_nine_percentile = values[math.floor(len(values) * 0.995)]

    for i in range(len(heatmap)):
        for j in range(len(heatmap[i])):
            if (heatmap[i][j] > ninty_nine_percentile):
                heatmap[i][j] = ninty_nine_percentile

    # np.max is from: Deep Learning with Python, Third edition. Chapter: 10, Displaying the class activation heatmap
    heatmap = (heatmap / np.max(heatmap)) * 255.0
    return heatmap

### The below cell is strongly based on Deep Learning with Python, Third edition. Chapter: 10, Visualizing heatmaps of class activation

In [6]:
import matplotlib.cm as cm

def superimpose(img_array, heatmap):
    # Uses the "jet" colormap to recolorize the heatmap
    jet = cm.get_cmap("jet")

    jet_colors = jet(np.arange(256))[:, :3]
    jet_colors-=[0,0,0.5]

    # Convertion to int is from: https://www.w3schools.com/python/numpy/numpy_data_types.asp (Converting Data Type on Existing Arrays)
    jet_heatmap = jet_colors[(np.round(heatmap)).astype('i')]

    # Superimposes the heatmap and the original image
    superimposed_img = jet_heatmap / 3.0 + img_array / 3.0
    superimposed_img = keras.utils.array_to_img(superimposed_img)
    return superimposed_img

# Cellene over kommer tilnærmet direkte fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]

In [7]:
import imageio
import gradio as gr

from preprocessing import preprocess_image_tensor

def classify_image(img: Image.Image, checkbox_group):
    print(img)
    # Resize to the input image to what MobileNet expects
    # img_resized = img.resize(IMG_RES)
    arr = np.array(img)
    print(img)
    # Check the color channels, so we can take both grayscale, RGB, and RGBA images as input.
    if arr.ndim == 2:
        arr = np.stack([arr] * 3, axis=-1)
    elif arr.shape[-1] == 4:
        arr = arr[..., :3]

    arr = arr / 255.0

    # Linjen under er basert på: https://www.tensorflow.org/api_docs/python/tf/keras/ops/convert_to_tensor
    # arr = keras.ops.convert_to_tensor(arr, dtype="float32")

    arr = preprocess_image_tensor(arr, IMG_RES[0])
    
    
    # Kodelinjen under kommer delvis fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
    # Kodelinjen i DAT255-prosjektet kommer i høyeste grad herfra: https://github.com/HVL-ML/DAT255/blob/main/notebooks/03_advanced_image_classification.ipynb.
    arr = keras.ops.expand_dims(arr, 0)

    # Linjen under er delvis basert på: https://www.gradio.app/docs/gradio/checkboxgroup
    return_array = [gr.Image(),gr.Image(),gr.Label(),gr.Label(),gr.Image(),gr.Image(),gr.Image(),gr.Image(),gr.Label(),gr.Label(),gr.Image(),gr.Image(),gr.Label(),gr.Label(), img, gr.CheckboxGroup()]

    if "Augment.keras (fra John Oscar)" in checkbox_group:
        model = model1
        # Predict!
        preds = model.predict(arr, verbose=0)
        

        # Kodelinjen under kommer delvis fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
        heatmap = get_heatmap(arr, "lvl1", model)
        
        # The below line is based on Deep Learning with Python, Third edition. Chapter: 10, Visualizing heatmaps of class activation
        superimposed = superimpose(arr[0], heatmap)

        # Line below is based on: https://pillow.readthedocs.io/en/stable/reference/Image.html
        superimposed_lvl2 = Image.new(mode="RGB", size=IMG_RES)
        if preds["lvl1"][0][0] > 0.5:
            # Kodelinjen under kommer delvis fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
            heatmap_lvl2 = get_heatmap(arr, "lvl2", model)
            # The below line is based on Deep Learning with Python, Third edition. Chapter: 10, Visualizing heatmaps of class activation
            superimposed_lvl2 = superimpose(arr[0], heatmap_lvl2)

        # Line below is based on https://stackoverflow.com/questions/57253048/scipy-misc-has-no-attribute-imsave and https://www.geeksforgeeks.org/python/getting-started-with-imageio-library-in-python/
        # imageio.imwrite("Neinieg.png", superimposed)

        car_model_preds = preds["lvl2"][0]
        return_array[0:4] = [superimposed, superimposed_lvl2, {"Tesla": preds["lvl1"][0][0], "Ikke-Tesla": 1-preds["lvl1"][0][0]}, {'3 2017–2023': car_model_preds[0], '3 2024–nå': car_model_preds[1], 'S 2012–2015': car_model_preds[2], 'S 2016–nå': car_model_preds[3], 'X': car_model_preds[4], 'Y 2020–2024': car_model_preds[5], 'Y 2025-nå': car_model_preds[6]}]

    if "Train_hierarchical_cnn_global_aug_v2.keras (Super god)" in checkbox_group:
        model = model2
        # Predict!
        preds = model.predict(arr, verbose=0)
        print(preds)
        # Linjen under kommer delvis fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
        # heatmap = get_heatmap(arr, "lvl1", model)

        # =====
        # Koden under kommer i høyeste grad fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
        # Koden fra DAT255-prosjektet kommer i høyeste grad herfra: SmoothGrad: removing noise by adding noise, 2.2. Smoothing noisy gradients (https://arxiv.org/pdf/1706.03825)
        # og herfra: https://numpy.org/doc/2.1/reference/random/generated/numpy.random.Generator.normal.html#numpy.random.Generator.normal

        rng = np.random.default_rng()
        finished_heatmap_lvl1 = np.zeros(arr[0].shape[:2])
        N = 50.0
        for n in range(int(N)):
            noise_map = rng.normal(0, 0.20, arr[0].shape)
            noisy_img = arr * noise_map
            current_heatmap = get_heatmap(noisy_img, "lvl1", model)
            finished_heatmap_lvl1 += current_heatmap
        finished_heatmap_lvl1 /= N
        # Kodelinjen under kommer fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse].
        # Idéen til den kodelinjen kommer herfra: SmoothGrad: removing noise by adding noise, Capping outlying values (https://arxiv.org/pdf/1706.03825)
        finished_heatmap_lvl1 = cap_heatmap(finished_heatmap_lvl1)

        # Koden over kommer i høyeste grad fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
        # Koden fra DAT255-prosjektet kommer i høyeste grad herfra: SmoothGrad: removing noise by adding noise, 2.2. Smoothing noisy gradients (https://arxiv.org/pdf/1706.03825)
        # og herfra: https://numpy.org/doc/2.1/reference/random/generated/numpy.random.Generator.normal.html#numpy.random.Generator.normal
        # =====

        # The below line is based on Deep Learning with Python, Third edition. Chapter: 10, Visualizing heatmaps of class activation
        superimposed = superimpose(arr[0], finished_heatmap_lvl1)

        # Line below is based on: https://pillow.readthedocs.io/en/stable/reference/Image.html
        superimposed_lvl2 = Image.new(mode="RGB", size=IMG_RES)
        if preds["lvl1"][0][0] > 0.5:
            # Linjen under kommer delvis fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
            # heatmap_lvl2 = get_heatmap(arr, "lvl2", model)

            # =====
            # Koden under kommer i høyeste grad fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
            # Koden fra DAT255-prosjektet kommer i høyeste grad herfra: SmoothGrad: removing noise by adding noise, 2.2. Smoothing noisy gradients (https://arxiv.org/pdf/1706.03825)
            # og herfra: https://numpy.org/doc/2.1/reference/random/generated/numpy.random.Generator.normal.html#numpy.random.Generator.normal
            
            rng = np.random.default_rng()
            finished_heatmap_lvl2 = np.zeros(arr[0].shape[:2])
            N = 50.0
            for n in range(int(N)):
                noise_map = rng.normal(0, 0.20, arr[0].shape)
                noisy_img = arr * noise_map
                # Argmax under er basert på: https://numpy.org/doc/2.2/reference/generated/numpy.argmax.html
                current_heatmap = get_heatmap(noisy_img, "lvl2", model, np.argmax(preds["lvl2"], 1)[0])
                finished_heatmap_lvl2 += current_heatmap
            finished_heatmap_lvl2 /= N
            # Kodelinjen under kommer fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse].
            # Idéen til den kodelinjen kommer herfra: SmoothGrad: removing noise by adding noise, Capping outlying values (https://arxiv.org/pdf/1706.03825)
            finished_heatmap_lvl2 = cap_heatmap(finished_heatmap_lvl2)

            # Koden over kommer i høyeste grad fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
            # Koden fra DAT255-prosjektet kommer i høyeste grad herfra: SmoothGrad: removing noise by adding noise, 2.2. Smoothing noisy gradients (https://arxiv.org/pdf/1706.03825)
            # og herfra: https://numpy.org/doc/2.1/reference/random/generated/numpy.random.Generator.normal.html#numpy.random.Generator.normal
            # =====

            # The below line is based on Deep Learning with Python, Third edition. Chapter: 10, Visualizing heatmaps of class activation
            superimposed_lvl2 = superimpose(arr[0], finished_heatmap_lvl2)

        # Line below is based on https://stackoverflow.com/questions/57253048/scipy-misc-has-no-attribute-imsave and https://www.geeksforgeeks.org/python/getting-started-with-imageio-library-in-python/
        # imageio.imwrite("Neinieg.png", superimposed)

        car_model_preds = preds["lvl2"][0]
        # Convertion to int is from: https://www.w3schools.com/python/numpy/numpy_data_types.asp (Converting Data Type on Existing Arrays)
        return_array[4:10] = [superimposed, np.round(finished_heatmap_lvl1).astype('i'), superimposed_lvl2, np.round(finished_heatmap_lvl2).astype('i'), {"Tesla": preds["lvl1"][0][0], "Ikke-Tesla": 1-preds["lvl1"][0][0]}, {'3 2017–2023': car_model_preds[0], '3 2024–nå': car_model_preds[1], 'S 2012–2015': car_model_preds[2], 'S 2016–nå': car_model_preds[3], 'X': car_model_preds[4], 'Y 2020–2024': car_model_preds[5], 'Y 2025-nå': car_model_preds[6]}]
    
    if "Horribel modell" in checkbox_group:
        model = model3
        # Predict!
        preds = model.predict(arr, verbose=0)

        # Kodelinjen under kommer delvis fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
        heatmap = get_heatmap(arr, "lvl1", model)
        
        # The below line is based on Deep Learning with Python, Third edition. Chapter: 10, Visualizing heatmaps of class activation
        superimposed = superimpose(arr[0], heatmap)

        # Line below is based on: https://pillow.readthedocs.io/en/stable/reference/Image.html
        superimposed_lvl2 = Image.new(mode="RGB", size=IMG_RES)
        if preds["lvl1"][0][0] > 0.5:
            # Kodelinjen under kommer delvis fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
            heatmap_lvl2 = get_heatmap(arr, "lvl2", model)
            # The below line is based on Deep Learning with Python, Third edition. Chapter: 10, Visualizing heatmaps of class activation
            superimposed_lvl2 = superimpose(arr[0], heatmap_lvl2)
        # Line below is based on https://stackoverflow.com/questions/57253048/scipy-misc-has-no-attribute-imsave and https://www.geeksforgeeks.org/python/getting-started-with-imageio-library-in-python/
        # imageio.imwrite("Neinieg.png", superimposed)

        car_model_preds = preds["lvl2"][0]
        return_array[10:14] = [superimposed, superimposed_lvl2, {"Tesla": preds["lvl1"][0][0], "Ikke-Tesla": 1-preds["lvl1"][0][0]}, {'3 2017–2023': car_model_preds[0], '3 2024–nå': car_model_preds[1], 'S 2012–2015': car_model_preds[2], 'S 2016–nå': car_model_preds[3], 'X': car_model_preds[4], 'Y 2020–2024': car_model_preds[5], 'Y 2025-nå': car_model_preds[6]}]
    return return_array



/media/d/skole/dat191/visual-vehicle-recognition-varying-lighting-conditions/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Cellen under er basert på: https://github.com/gradio-app/gradio/issues/2066 og https://discuss.huggingface.co/t/how-to-programmatically-enable-or-disable-components/52350

In [8]:
def change_visibility(checkbox_group, model_1_label1, model_1_label2, model_2_label1, model_2_label2, model_3_label1, model_3_label2):
    return_array = [gr.Image(),gr.Image(),gr.Label(),gr.Label(),gr.Markdown(),gr.Image(),gr.Image(),gr.Image(),gr.Image(),gr.Label(),gr.Label(),gr.Markdown(),gr.Image(),gr.Image(),gr.Label(),gr.Label(),gr.Markdown()]
    if "Augment.keras (fra John Oscar)" in checkbox_group:
        try:
            model_1_label1["Tesla"]
            # The line below is based on: https://www.geeksforgeeks.org/python/python-get-key-with-maximum-value-in-dictionary/
            predicted_model = max(model_1_label2, key=model_1_label2.get)
            return_array[0:5] = [gr.Image(label="Heatmap som viser hvorfor Tesla", visible=True), gr.Image(visible=True, label="Heatmap som viser hvorfor " + predicted_model), gr.Label(visible=True), gr.Label(visible=True), gr.Markdown(visible=True)]
        except:
            return_array[0:5] = [gr.Image(label="Heatmap som viser hvorfor ikke-Tesla", visible=True), gr.Image(visible=False), gr.Label(visible=True), gr.Label(visible=False), gr.Markdown(visible=True)]
    else: 
        return_array[0:5] = [gr.Image(visible=False), gr.Image(visible=False), gr.Label(visible=False), gr.Label(visible=False), gr.Markdown(visible=False)]
    if "Train_hierarchical_cnn_global_aug_v2.keras (Super god)" in checkbox_group:
        try:
            model_2_label1["Tesla"]
            # The line below is based on: https://www.geeksforgeeks.org/python/python-get-key-with-maximum-value-in-dictionary/
            predicted_model = max(model_2_label2, key=model_2_label2.get)
            return_array[5:12] = [gr.Image(label="Heatmap som viser hvorfor Tesla", visible=True), gr.Image(label="Heatmap som viser hvorfor Tesla", visible=True), gr.Image(visible=True, label="Heatmap som viser hvorfor " + predicted_model), gr.Image(visible=True, label="Heatmap som viser hvorfor " + predicted_model), gr.Label(visible=True), gr.Label(visible=True), gr.Markdown(visible=True)]
        except:
            return_array[5:12] = [gr.Image(label="Heatmap som viser hvorfor ikke-Tesla", visible=True), gr.Image(label="Heatmap som viser hvorfor ikke-Tesla", visible=True), gr.Image(visible=False), gr.Image(visible=False), gr.Label(visible=True), gr.Label(visible=False), gr.Markdown(visible=True)]
    else: 
        return_array[5:12] = [gr.Image(visible=False), gr.Image(visible=False), gr.Image(visible=False), gr.Image(visible=False), gr.Label(visible=False), gr.Label(visible=False), gr.Markdown(visible=False)]
    if "Horribel modell" in checkbox_group:
        try:
            model_3_label1["Tesla"]
            # The line below is based on: https://www.geeksforgeeks.org/python/python-get-key-with-maximum-value-in-dictionary/
            predicted_model = max(model_3_label2, key=model_3_label2.get)
            return_array[12:17] = [gr.Image(label="Heatmap som viser hvorfor Tesla", visible=True), gr.Image(visible=True, label="Heatmap som viser hvorfor " + predicted_model), gr.Label(visible=True), gr.Label(visible=True), gr.Markdown(visible=True)]
        except:
            return_array[12:17] = [gr.Image(label="Heatmap som viser hvorfor ikke-Tesla", visible=True), gr.Image(visible=False), gr.Label(visible=True), gr.Label(visible=False), gr.Markdown(visible=True)]
    else: 
        return_array[12:17] = [gr.Image(visible=False), gr.Image(visible=False), gr.Label(visible=False), gr.Label(visible=False), gr.Markdown(visible=False)]
    return return_array

## Set up the Gradio interface

Check the [documentation](https://www.gradio.app/docs) for the various things we can add here.

In [9]:
# Example images
examples = [
    "https://cdn.britannica.com/79/232779-050-6B0411D7/German-Shepherd-dog-Alsatian.jpg",
    "https://cdn.britannica.com/41/126641-050-E1CA0E61/cat-suns-hill-Parthenon-Athens-Greece-Acropolis.jpg",
]

with gr.Blocks(title="Multiattributt visuell kjøretøygjenkjenning under varierende lysforhold") as demo:
    gr.Markdown("## Multiattributt visuell kjøretøygjenkjenning under varierende lysforhold")
    gr.Markdown(
        "Last opp et bilde av et kjøretøy eller velg et eksempel under. "
    )
    with gr.Row():
        components = [[0,0,0,0,0],[0,0,0,0,0,0,0],[0,0,0,0,0],0,0]
        components[3] = gr.Image(type="pil", label="Opplastet bilde")
        # Linjen under er basert på: https://www.gradio.app/docs/gradio/checkboxgroup
        components[4] = gr.CheckboxGroup(choices=["Augment.keras (fra John Oscar)", "Train_hierarchical_cnn_global_aug_v2.keras (Super god)", "Horribel modell"])

    # Linjen under er basert på: https://www.gradio.app/docs/gradio/blocks
    components[0][4] = gr.Markdown("Prediksjonene til Augment.keras (fra John Oscar):", visible=False)
    with gr.Row():
        # De to linjene under er delvis basert på: https://github.com/gradio-app/gradio/issues/10394
        components[0][0] = gr.Image(type="pil", label="Heatmap", format="png", visible=False)
        components[0][1] = gr.Image(type="pil", label="Heatmap", format="png", visible=False)
        # Linjen under er basert på: https://www.gradio.app/4.44.1/guides/controlling-layout
        with gr.Column():
            components[0][2] = gr.Label(num_top_classes=1, label="Bilmerke", visible=False)
            components[0][3] = gr.Label(num_top_classes=3, label="Bilmodell", visible=False)

    # Linjen under er basert på: https://www.gradio.app/docs/gradio/blocks
    components[1][6] = gr.Markdown("Prediksjonene til Train_hierarchical_cnn_global_aug_v2.keras (Super god):", visible=False)
    with gr.Row():
        # Linjen under er basert på: https://www.gradio.app/4.44.1/guides/controlling-layout
        with gr.Column():
            # De to linjene under er delvis basert på: https://github.com/gradio-app/gradio/issues/10394
            components[1][0] = gr.Image(type="pil", label="Heatmap", format="png", visible=False)
            components[1][1] = gr.Image(type="pil", label="Heatmap", format="png", visible=False)
        # Linjen under er basert på: https://www.gradio.app/4.44.1/guides/controlling-layout
        with gr.Column():
            # De to linjene under er delvis basert på: https://github.com/gradio-app/gradio/issues/10394
            components[1][2] = gr.Image(type="pil", label="Heatmap", format="png", visible=False)
            components[1][3] = gr.Image(type="pil", label="Heatmap", format="png", visible=False)
        # Linjen under er basert på: https://www.gradio.app/4.44.1/guides/controlling-layout
        with gr.Column():
            components[1][4] = gr.Label(num_top_classes=1, label="Bilmerke", visible=False)
            components[1][5] = gr.Label(num_top_classes=3, label="Bilmodell", visible=False)

    # Linjen under er basert på: https://www.gradio.app/docs/gradio/blocks
    components[2][4] = gr.Markdown("Prediksjonene til Horribel modell:", visible=False)
    with gr.Row():
        # De to linjene under er delvis basert på: https://github.com/gradio-app/gradio/issues/10394
        components[2][0] = gr.Image(type="pil", label="Heatmap", format="png", visible=False)
        components[2][1] = gr.Image(type="pil", label="Heatmap", format="png", visible=False)
        # Linjen under er basert på: https://www.gradio.app/4.44.1/guides/controlling-layout
        with gr.Column():
            components[2][2] = gr.Label(num_top_classes=1, label="Bilmerke", visible=False)
            components[2][3] = gr.Label(num_top_classes=3, label="Bilmodell", visible=False)

    classify_btn = gr.Button("Classify", variant="primary")
    classify_btn.click(fn=classify_image, inputs=[components[3],components[4]], outputs=[components[0][0],components[0][1],components[0][2],components[0][3],components[1][0],components[1][1],components[1][2],components[1][3],components[1][4],components[1][5],components[2][0],components[2][1],components[2][2],components[2][3], components[3], components[4]])
    # Linjen under er basert på: https://github.com/gradio-app/gradio/issues/2066 og https://discuss.huggingface.co/t/how-to-programmatically-enable-or-disable-components/52350
    classify_btn.click(fn=change_visibility, inputs=[components[4], components[0][2], components[0][3], components[1][4], components[1][5], components[2][2], components[2][3]], outputs=[components[0][0], components[0][1], components[0][2], components[0][3], components[0][4], components[1][0], components[1][1], components[1][2], components[1][3], components[1][4], components[1][5], components[1][6], components[2][0], components[2][1], components[2][2], components[2][3], components[2][4]])

    examples = gr.Examples(
        examples=examples,
        inputs=components[3],
        # outputs=output,
        # fn=classify_image,
        # cache_examples=True
    )

Now we can run it:

In [ ]:
demo.launch(share=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


<PIL.Image.Image image mode=RGB size=3024x4032 at 0x73986812CB90>
<PIL.Image.Image image mode=RGB size=3024x4032 at 0x73986812CB90>


I0000 00:00:1774624488.092428    9368 cuda_dnn.cc:529] Loaded cuDNN version 92000


{'lvl1': array([[1.]], dtype=float32), 'lvl2': array([[9.8151129e-01, 1.8643775e-06, 1.8116733e-12, 1.5820935e-05,
        2.2834164e-04, 1.8226750e-02, 1.5924792e-05]], dtype=float32)}


I0000 00:00:1774624489.594174    9432 cuda_solvers.cc:178] Creating GpuSolver handles for stream 0x576ecbd85750
/tmp/ipykernel_9267/3145468918.py:5: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  jet = cm.get_cmap("jet")
